# GIRF - Dyn-method with triangular pulses

Estimate the gradient impulse response function using the Dyn-method with triangular pulses

### Imports

In [ ]:
import tempfile
from collections.abc import Sequence
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import torch
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryCartesian

from mrseq.scripts.girf_dyn_triangle import main as create_seq
from mrseq.utils import sys_defaults

### Settings
We are going to use a numerical phantom with a matrix size of 128 x 128. 

In [ ]:
image_matrix_size = [128, 128]

tmp = tempfile.TemporaryDirectory()
fname_mrd = Path(tmp.name) / 'girf_dyn_triangle.mrd'

### Create the digital phantom

We use the standard Brainweb phantom from [MRzero](https://github.com/MRsources/MRzero-Core).

In [ ]:
phantom = mr0.util.load_phantom(image_matrix_size)
phantom.T1[:] = 400e-3
phantom.T2[:] = 40e-3
phantom.B1[:] = 1.0
phantom.B0[:] = 0.0

### Create the GIRF Dyn sequence

To create the GIRF Dyn sequence, we use the previously imported [girf_dyn_triangle script](../src/mrseq/scripts/girf_dyn_triangle.py).

In [ ]:
sequence, fname_seq = create_seq(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
)

### Simulate the sequence

Now, we pass the sequence and the phantom to the MRzero simulation and save the simulated signal as an (ISMR)MRD file.

In [ ]:
mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e-1)
mr0.sig_to_mrd(fname_mrd, signal, sequence)

### Estimate GIRF

#### Sort data and compress coils

In [ ]:
kdata = KData.from_file(fname_mrd, trajectory=KTrajectoryCartesian())
idx = np.lexsort(
    (
        kdata.header.acq_info.idx.phase.squeeze(),
        kdata.header.acq_info.idx.repetition.squeeze(),
        kdata.header.acq_info.idx.average.squeeze(),
    )
)
kdata_sorted = kdata[torch.as_tensor(idx)]
kdata_sorted = kdata_sorted.rearrange(
    '(avg rep ph) ... -> avg rep ph ...',
    rep=kdata.header.acq_info.idx.repetition.max() + 1,
    avg=kdata.header.acq_info.idx.average.max() + 1,
    ph=kdata.header.acq_info.idx.phase.max() + 1,
)
kdat_single_coil = kdata_sorted.compress_coils(n_compressed_coils=1).data.squeeze()

#### Get information about the scan from the seq file

In [ ]:
adc_duration = sequence.get_definition('AdcDuration')
dwell_time = sequence.get_definition('DwellTime')
slice_pos = sequence.get_definition('SlicePos')[0]
gamma = sequence.system.gamma * 2 * torch.pi
rise_times = sequence.get_definition('RiseTimes')
slew_rate = sequence.get_definition('SlewRate') / sequence.system.gamma
g_delay = sequence.get_definition('AdcDelay')
g_amplitude_coeff = sequence.get_definition('GradAmplitudeCoeff')

output_delay = 2

#### Unwrapped measured gradient shapes

In [ ]:
def unwrap_phase_difference(data: torch.Tensor) -> torch.Tensor:
    """Return doubly-unwrapped phase of polarity[0] - polarity[1].

    Input shape:  ``(n_avg, 2, n_axes, n_rise, n_samples)``
    Output shape: ``(n_avg, n_axes, n_rise, n_samples)``
    """
    diff = data[:, 0, ...].angle() - data[:, 1, ...].angle()
    return torch.from_numpy(np.unwrap(np.unwrap(diff.numpy())))

In [ ]:
def phase_to_gradient(
    phase_mean: torch.Tensor,
    phase_std: torch.Tensor,
    slice_pos: float,
    gamma: float,
    dwell_time: float,
    output_delay: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Convert unwrapped-phase arrays to gradient waveforms via finite difference."""
    scale = slice_pos * gamma * dwell_time
    grad_mean = torch.roll(phase_mean.diff(dim=-1) / scale, output_delay, dims=-1)
    grad_std = phase_std.diff(dim=-1) / scale
    return grad_mean, grad_std

In [ ]:
kdata_sorted.data.shape

In [ ]:
plt.figure()
plt.plot(kdata_sorted.data[0, 1, 0, 0, 0].T.abs())

plt.figure()
plt.plot(kdata_sorted.data[0, 1, 0, 0, 0].T.angle())

In [ ]:
n_samples = round(adc_duration / dwell_time)
time = torch.linspace(0.0, adc_duration, n_samples)

phase = unwrap_phase_difference(kdat_single_coil)
phase_mean = phase.mean(dim=0)
phase_std = phase.std(dim=0)

grad_output_mean, grad_output_std = phase_to_gradient(phase_mean, phase_std, slice_pos, gamma, dwell_time, output_delay)

In [ ]:
# Plot unwrapped phase (k-space trajectories) for each gradient axis
grad_axes = ['x', 'y', 'z']
fig, axes = plt.subplots(len(grad_axes), 1, sharex=True)  # (avg, ax, rise, samp)
for ax_idx, (ax, label) in enumerate(zip(axes, grad_axes, strict=True)):
    for rise_idx in range(phase.shape[2]):
        ax.plot(time, phase_mean.numpy()[ax_idx, rise_idx, :])
    ax.set_title(f'K-space — {label}')
    ax.set_xlabel('Time [s]')
    ax.grid()
fig.tight_layout()
plt.show()

#### Create ideal gradient triangles

In [ ]:
def build_input_triangles(
    rise_times: Sequence[float],
    slew_rate: float,
    g_delay: float,
    enumerate_coeff: Sequence[float],
    dwell_time: float,
    n_samples: int,
) -> torch.Tensor:
    """Build ideal triangular input waveforms.

    Returns
    -------
    Tensor of shape ``(n_rise, n_samples)`` [T/m].
    """
    triangles = torch.zeros(len(rise_times), n_samples)
    for i, rise_time in enumerate(rise_times):
        n_rise = round(rise_time / dwell_time)
        n_pre = round(g_delay / dwell_time)
        amplitude = enumerate_coeff[0] * slew_rate * rise_time

        slope_up = torch.linspace(0.0, amplitude, n_rise + 1)
        slope_down = torch.linspace(amplitude, 0.0, n_rise + 1)

        triangles[i, n_pre : n_pre + n_rise + 1] = slope_up
        triangles[i, n_pre + n_rise + 1 : n_pre + 2 * n_rise + 1] = slope_down[1:]

    return triangles

In [ ]:
grad_input = build_input_triangles(rise_times, slew_rate, g_delay, g_amplitude_coeff, dwell_time, n_samples)

#### Compare ideal and measured gradient triangles

In [ ]:
# Overlay ideal input triangles against measured X-axis output
plt.figure()
for i in range(grad_input.shape[0]):
    line = plt.plot(time * 1e3, grad_input[i, :], '-')
    plt.plot(time[:-1] * 1e3, grad_output_mean[0, i, :], color=line[0].get_color(), linestyle='dashed')

plt.xlim((0.0, 0.4))
plt.xlabel('Time [ms]')
plt.ylabel('Gradient Strength [T/m]')
plt.title('Input (solid) vs Measurement (dashed)')
plt.grid()
plt.tight_layout()
plt.show()

#### Calculate GIRF and plot

In [ ]:
def estimate_gmtf(
    grad_input: torch.Tensor,
    grad_output_mean: torch.Tensor,
) -> torch.Tensor:
    """Least-squares GMTF estimate over all rise times.

    Parameters
    ----------
    grad_input:
        Ideal waveforms, shape ``(n_rise, n_samples)``.
    grad_output_mean:
        Measured waveforms, shape ``(n_axes, n_rise, n_samples-1)``.

    Returns
    -------
    Complex tensor of shape ``(n_axes, n_freq)``.
    """
    n_fft_in = 2 * (grad_input.shape[-1] - 1)
    n_fft_out = 2 * grad_output_mean.shape[-1]

    in_spec = torch.fft.fftshift(torch.fft.fft(grad_input, n=n_fft_in, dim=-1), dim=-1)
    out_spec = torch.fft.fftshift(torch.fft.fft(grad_output_mean, n=n_fft_out, dim=-1), dim=-1)

    # Least-squares: sum_rt conj(H_in) * H_out  /  sum_rt |H_in|^2
    numerator = (in_spec.conj() * out_spec).sum(dim=-2)  # sum over rise dim
    denominator = (in_spec.abs() ** 2).sum(dim=-2)
    return numerator / denominator

In [ ]:
gmtf = estimate_gmtf(grad_input, grad_output_mean)

In [ ]:
# Plot magnitude and phase of the GMTF for all axes
grad_axes = ['x', 'y', 'z']

n_freq = n_samples - 1
frequency_vector = (1.0 / dwell_time) * torch.arange(-n_freq, n_freq) / (2 * n_samples)
half = len(frequency_vector) // 2

fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, figsize=(10, 5))
for k, label in enumerate(grad_axes):
    ax1.plot(frequency_vector[half:] * 1e-3, abs(gmtf.numpy()[k, half:]), label=label)
    ax2.plot(frequency_vector[half:] * 1e-3, np.unwrap(np.angle(gmtf.numpy()[k, half:])), label=label)

ax1.set(xlabel='Frequency (kHz)', ylabel='|GMTF|', title='Magnitude', xlim=(0, 10), ylim=(0.8, 1.025))
ax2.set(xlabel='Frequency (kHz)', ylabel='arg(GMTF) (rad)', title='Phase', xlim=(0, 10), ylim=(-0.1, 0.1))

for ax in (ax1, ax2):
    ax.legend()
    ax.grid(visible=True, which='major', color='#666666')
    ax.minorticks_on()
    ax.grid(visible=True, which='minor', color='#999999', linestyle='-', alpha=0.2)

fig.tight_layout()
plt.show()